In [4]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain_huggingface import HuggingFaceEmbeddings  # Note: Can also be imported from langchain_huggingface
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiQueryRetriever  # Updated import

At its core, the MultiQueryRetriever is a tool designed to solve a major limitation in standard vector search: it fixes bad user queries.

When you ask a typical Vector DB setup a question, it converts your exact wording into a vector embedding and looks for matches. If your query uses slightly different vocabulary than the document (even if the meaning is identical), the system can easily miss the right answer.

The MultiQueryRetriever acts as an automated prompt-engineering layer. Instead of searching with just your single query, it uses a Large Language Model (LLM) to automatically rewrite your question from multiple perspectives, searches for all of them, and combines the results.

In [ ]:
import os 
os.environ["HUGGINGFACEHUB_ACCESS_TOKEN"] = ""

In [7]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [12]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings

embedding_model = embedding = HuggingFaceEndpointEmbeddings(
    model= 'sentence-transformers/all-MiniLM-L6-v2',
    task= 'feature-extraction'
)

vectorstore = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model
)

#This creates the FAISS vectorstore in memory only — nothing is stored on disk.
# FAISS stands for Facebook AI Similarity Search, and it is a library for efficient similarity search and clustering of dense vectors. It is commonly used for tasks like nearest neighbor search in high-dimensional spaces, which is essential for applications like document retrieval, image search, and recommendation systems.
# In this code, we are using FAISS to create a vector store from a list of documents. The `from_documents` method takes the documents and an embedding model to convert the documents into. 
# Yes, the FAISS vector store created here is in memory only, meaning that it is not persisted to disk. If you want to save the vector store for later use, you would need to implement additional code to serialize and save the FAISS index to disk.

In [13]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [14]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

In [16]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=model
)

In [17]:

# Query
query = "How to improve energy levels and maintain balance?"

In [18]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [19]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
******************************************************************************************************************************************************

--- Result 1 ---
Black holes bend spacetime and store immense gravitational energy.

--- Result 2 ---
Python balances readability with power, making it a popular system design language.

--- Result 3 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.